# Silver Layer — Aggregated Users

This notebook transforms PhonePe aggregated user JSON data from the Bronze layer into a clean, analytics-ready Silver Delta table.

## Sources

- Bronze table: `workspace.bronze.phonepe_raw_json`
- Metadata table: `workspace.silver.phonepe_file_metadata`

## Target

- User registration table: `workspace.silver.aggregated_users`

## Processing

- Select aggregated user files
- Inspect and parse the nested JSON structure
- Extract registered-user counts
- Preserve state, year, quarter, and source lineage
- Validate completeness, uniqueness, and numerical values

## Source limitation

The pinned PhonePe source version contains `registeredCount` but does not provide app-open or device-brand information in aggregated user files.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    LongType
)

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
USER_ACTIVITY_TABLE = "workspace.silver.aggregated_users"

# 34 country files + 1,224 state files
EXPECTED_SOURCE_FILES = 1_258

print(f"Bronze source   : {BRONZE_TABLE}")
print(f"Metadata source : {METADATA_TABLE}")
print(f"Silver target   : {USER_ACTIVITY_TABLE}")
print(f"Expected files  : {EXPECTED_SOURCE_FILES:,}")

Bronze source   : workspace.bronze.phonepe_raw_json
Metadata source : workspace.silver.phonepe_file_metadata
Silver target   : workspace.silver.aggregated_users
Expected files  : 1,258


In [0]:
bronze_df = spark.table(BRONZE_TABLE).alias("b")

user_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "aggregated") &
        (F.col("domain") == "user")
    )
    .alias("m")
)

user_source_df = (
    user_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

actual_source_files = (
    user_source_df
    .select("source_path")
    .distinct()
    .count()
)

print(f"Expected source files: {EXPECTED_SOURCE_FILES:,}")
print(f"Actual source files  : {actual_source_files:,}")

assert actual_source_files == EXPECTED_SOURCE_FILES, (
    f"File-count validation failed: expected "
    f"{EXPECTED_SOURCE_FILES:,}, found {actual_source_files:,}"
)

print("Aggregated user source validation passed.")

Expected source files: 1,258
Actual source files  : 1,258
Aggregated user source validation passed.


In [0]:
user_structure_check_df = (
    user_source_df
    .select(
        F.col("year"),

        F.get_json_object(
            F.col("raw_json"),
            "$.data.aggregated"
        ).alias("aggregated_user_metrics"),

        F.get_json_object(
            F.col("raw_json"),
            "$.data.usersByDevice[0]"
        ).alias("sample_device_record")
    )
    .groupBy("year")
    .agg(
        F.first(
            F.col("aggregated_user_metrics"),
            True
        ).alias("aggregated_user_metrics"),

        F.first(
            F.col("sample_device_record"),
            True
        ).alias("sample_device_record")
    )
    .orderBy("year")
)

display(user_structure_check_df)

year,aggregated_user_metrics,sample_device_record
2018,"{""registeredCount"":1200972}",null
2019,"{""registeredCount"":142792343}",null
2020,"{""registeredCount"":14049378}",null
2021,"{""registeredCount"":306887085}",null
2022,"{""registeredCount"":22199210}",null
2023,"{""registeredCount"":473707081}",null
2024,"{""registeredCount"":28578957}",null
2025,"{""registeredCount"":637393317}",null
2026,"{""registeredCount"":14095525}",null


In [0]:
aggregated_user_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),
    StructField(
        "data",
        StructType([
            StructField(
                "aggregated",
                StructType([
                    StructField(
                        "registeredCount",
                        LongType(),
                        True
                    )
                ]),
                True
            )
        ]),
        True
    ),
    StructField("responseTimestamp", LongType(), True)
])

print("Aggregated user JSON schema created successfully.")

Aggregated user JSON schema created successfully.


In [0]:
parsed_user_df = (
    user_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            aggregated_user_schema
        )
    )
)

parse_summary = (
    parsed_user_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col(
                    "parsed_json.data.aggregated.registeredCount"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_registered_count")
    )
    .first()
)

print(f"Source files                  : {parse_summary['source_files']:,}")
print(
    f"Files without registered count: "
    f"{parse_summary['files_without_registered_count']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_registered_count"] == 0, (
    "JSON parsing validation failed: some files do not "
    "contain registeredCount."
)

print("Aggregated user JSON parsing validation passed.")

Source files                  : 1,258
Files without registered count: 0
Aggregated user JSON parsing validation passed.


In [0]:
silver_users_df = (
    parsed_user_df
    .select(
        F.col("file_scope"),
        F.col("state_slug"),
        F.col("year").cast("int").alias("year"),
        F.col("quarter").cast("int").alias("quarter"),
        F.col("period"),

        F.col(
            "parsed_json.data.aggregated.registeredCount"
        ).cast("long").alias("registered_user_count"),

        F.col(
            "parsed_json.responseTimestamp"
        ).alias("response_timestamp_epoch_ms"),

        F.col("relative_path"),
        F.col("source_path"),
        F.col("source_commit"),
        F.col("content_sha256"),
        F.col("ingested_at_utc")
    )
)

print("Aggregated user records transformed successfully.")

display(
    silver_users_df
    .orderBy(
        "year",
        "quarter",
        "file_scope",
        "state_slug"
    )
    .limit(10)
)

Aggregated user records transformed successfully.


file_scope,state_slug,year,quarter,period,registered_user_count,response_timestamp_epoch_ms,relative_path,source_path,source_commit,content_sha256,ingested_at_utc
country,null,2018,1,2018-Q1,47483331,1785043997581,aggregated/user/country/india/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,2dc4e479e04e0b75b1f911aa67e7da55349247788b9e553be97973c922b447d9,2026-10-09T21:12:12.947Z
state,andaman-&-nicobar-islands,2018,1,2018-Q1,9292,1785044053151,aggregated/user/country/india/state/andaman-&-nicobar-islands/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/andaman-&-nicobar-islands/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,b450c73466cba8a64ca31783a5b60493145bf657db01a87e68069d275f745e2a,2026-10-09T21:12:12.947Z
state,andhra-pradesh,2018,1,2018-Q1,2979027,1785044053151,aggregated/user/country/india/state/andhra-pradesh/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/andhra-pradesh/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,019fea4652472fe5e08c7a9d4a0b9bd3c236fbfb3444655aac621a54cbd86af7,2026-10-09T21:12:12.947Z
state,arunachal-pradesh,2018,1,2018-Q1,26660,1785044053151,aggregated/user/country/india/state/arunachal-pradesh/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/arunachal-pradesh/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,bea1e48ae573488320bbc62146c20903a6020d83f6f34d6d94fe20b126d71532,2026-10-09T21:12:12.947Z
state,assam,2018,1,2018-Q1,449118,1785044053151,aggregated/user/country/india/state/assam/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/assam/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,e565cfff9f640305be161fbc2d35d8be4dc4e8ff4349031b05b8a148eeb52515,2026-10-09T21:12:12.947Z
state,bihar,2018,1,2018-Q1,2173873,1785044053151,aggregated/user/country/india/state/bihar/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/bihar/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,e9d93fb5d5434ce0fe555463e9b288119bf471af35970a2af174e03ec2efe60c,2026-10-09T21:12:12.947Z
state,chandigarh,2018,1,2018-Q1,84504,1785044053153,aggregated/user/country/india/state/chandigarh/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/chandigarh/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,bb84f979f0cc447423f3c848c28b12f567a7ac370f36a237e13ea81e4e9da015,2026-10-09T21:12:12.947Z
state,chhattisgarh,2018,1,2018-Q1,686395,1785044053152,aggregated/user/country/india/state/chhattisgarh/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/chhattisgarh/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,ab759d920dbef0cd176b3a2bb06218640dabb85736804c5011d25686530bd0fc,2026-10-09T21:12:12.947Z
state,dadra-&-nagar-haveli-&-daman-&-diu,2018,1,2018-Q1,46505,1785044053152,aggregated/user/country/india/state/dadra-&-nagar-haveli-&-daman-&-diu/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/user/country/india/state/dadra-&-nagar-haveli-&-daman-&-diu/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,8c96ee01a192595b5385665a53c6b7fb1d40da40e41dfa0dee6c81d96abb

In [0]:
user_quality_result = (
    silver_users_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("registered_user_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_user_counts"),

        F.sum(
            F.when(
                F.col("registered_user_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_user_counts"),

        F.min("registered_user_count")
            .alias("minimum_user_count"),

        F.max("registered_user_count")
            .alias("maximum_user_count")
    )
    .first()
)

duplicate_business_keys = (
    silver_users_df
    .groupBy(
        "file_scope",
        "state_slug",
        "year",
        "quarter"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Total Silver rows       : {user_quality_result['total_rows']:,}")
print(f"Missing user counts     : {user_quality_result['missing_user_counts']:,}")
print(f"Negative user counts    : {user_quality_result['negative_user_counts']:,}")
print(f"Minimum registered users: {user_quality_result['minimum_user_count']:,}")
print(f"Maximum registered users: {user_quality_result['maximum_user_count']:,}")
print(f"Duplicate business keys : {duplicate_business_keys:,}")

assert user_quality_result["total_rows"] == EXPECTED_SOURCE_FILES
assert user_quality_result["missing_user_counts"] == 0
assert user_quality_result["negative_user_counts"] == 0
assert duplicate_business_keys == 0

print("Silver aggregated-user quality checks passed.")

Total Silver rows       : 1,258
Missing user counts     : 0
Negative user counts    : 0
Minimum registered users: 862
Maximum registered users: 711,653,262
Duplicate business keys : 0
Silver aggregated-user quality checks passed.


In [0]:
(
    silver_users_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(USER_ACTIVITY_TABLE)
)

print(f"Silver Delta table created: {USER_ACTIVITY_TABLE}")

Silver Delta table created: workspace.silver.aggregated_users


In [0]:
saved_users_df = spark.table(USER_ACTIVITY_TABLE)

saved_user_summary = (
    saved_users_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period")
    )
    .first()
)

print(f"Saved rows              : {saved_user_summary['row_count']:,}")
print(f"Source files represented: {saved_user_summary['source_file_count']:,}")
print(f"First period            : {saved_user_summary['first_period']}")
print(f"Last period             : {saved_user_summary['last_period']}")

assert saved_user_summary["row_count"] == EXPECTED_SOURCE_FILES
assert saved_user_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_user_summary["first_period"] == "2018-Q1"
assert saved_user_summary["last_period"] == "2026-Q2"

print("Final aggregated-user Delta-table validation passed.")

Saved rows              : 1,258
Source files represented: 1,258
First period            : 2018-Q1
Last period             : 2026-Q2
Final aggregated-user Delta-table validation passed.


## Silver Aggregated User Model — Completion Summary

The `workspace.silver.aggregated_users` Delta table was successfully created.

### Validation results

- Source files represented: **1,258**
- Silver records: **1,258**
- Data coverage: **2018-Q1 to 2026-Q2**
- Missing registered-user values: **0**
- Negative registered-user values: **0**
- Duplicate business keys: **0**
- Minimum registered-user count: **862**
- Maximum registered-user count: **711,653,262**

### Table grain

Each record represents one geographical scope for one year and quarter:

- Country-quarter, or
- State-quarter

### Source-data limitation

The pinned PhonePe source version provides only `registeredCount`. It does not provide app-open or device-brand metrics in the aggregated user files.

### Analytical interpretation

`registered_user_count` is a cumulative stock metric. It should not be summed across quarters. Quarterly user additions and growth rates will be derived later in the Gold/dbt layer using window functions.